In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

## Load Intermediate Features Dataset

In [2]:
# Load intermediate features
features = pd.read_csv("../intermediate/features.csv", parse_dates=["signup_date"])

print("Intermediate Features Dataset")
print("=" * 60)
print(f"Shape: {features.shape}")
print(f"\nColumns: {list(features.columns)}")
print(f"\nData types:\n{features.dtypes}")

Intermediate Features Dataset
Shape: (5000, 29)

Columns: ['customer_id', 'signup_date', 'country', 'age', 'gender', 'subscription_tier', 'monthly_fee', 'account_age_days', 'is_premium', 'age_group', 'country_region', 'total_sessions', 'avg_sessions_per_day', 'total_usage_minutes', 'avg_session_duration', 'avg_features_per_session', 'total_errors', 'days_active', 'days_since_last_activity', 'usage_consistency_std', 'usage_trend', 'total_tickets', 'avg_resolution_hours', 'unresolved_tickets', 'technical_tickets_pct', 'high_priority_tickets', 'avg_satisfaction', 'days_since_last_ticket', 'churned']

Data types:
customer_id                          int64
signup_date                 datetime64[ns]
country                             object
age                                float64
gender                              object
subscription_tier                   object
monthly_fee                        float64
account_age_days                     int64
is_premium                           in

In [3]:
# Show first few rows
features.head()

,customer_id,signup_date,country,age,gender,subscription_tier,monthly_fee,account_age_days,is_premium,age_group,country_region,total_sessions,avg_sessions_per_day,total_usage_minutes,avg_session_duration,avg_features_per_session,total_errors,days_active,days_since_last_activity,usage_consistency_std,usage_trend,total_tickets,avg_resolution_hours,unresolved_tickets,technical_tickets_pct,high_priority_tickets,avg_satisfaction,days_since_last_ticket,churned
0,1,2023-10-17,Germany,62.00,F,Basic,13.00,441,0,51+,Europe,349,5.63,1745.06,28.15,8.13,26,62,0,2.65,0.83,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
1,2,2022-04-25,UK,43.50,M,Premium,30.48,981,1,36-50,Europe,403,5.30,2515.99,33.11,7.57,39,76,0,2.88,0.67,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
2,3,2022-01-26,Brazil,53.00,Unknown,Basic,11.59,1070,0,51+,South America,421,5.40,2312.93,29.65,7.53,42,78,0,2.49,0.59,4.00,20.02,1.00,25.00,2.00,0.00,70.00,0
3,4,2024-01-30,Canada,26.00,M,Basic,11.97,336,0,26-35,North America,147,5.25,662.64,23.67,6.86,12,28,4,2.24,-0.04,1.00,12.11,0.00,100.00,0.00,4.00,286.00,0
4,5,2022-10-09,Spain,44.00,F,Basic,11.23,814,0,36-50,Europe,263,4.61,1803.58,31.64,7.09,37,57,0,2.60,-0.13,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0


---

## Step 1: Remove Columns Not Needed for ML

### Columns to Remove:
1. **customer_id** - Just an identifier, not predictive
2. **signup_date** - Already captured in account_age_days
3. **country** - Already captured in country_region

These are either identifiers or redundant with other features

In [4]:
# Create a copy for the final dataset
ml_data = features.copy()

# Drop columns not needed for ML
columns_to_drop = ["customer_id", "signup_date", "country"]
ml_data = ml_data.drop(columns=columns_to_drop)

print(f"Dropped columns: {columns_to_drop}")
print(f"Shape after dropping: {ml_data.shape}")
print(f"\nRemaining columns: {list(ml_data.columns)}")

Dropped columns: ['customer_id', 'signup_date', 'country']
Shape after dropping: (5000, 26)

Remaining columns: ['age', 'gender', 'subscription_tier', 'monthly_fee', 'account_age_days', 'is_premium', 'age_group', 'country_region', 'total_sessions', 'avg_sessions_per_day', 'total_usage_minutes', 'avg_session_duration', 'avg_features_per_session', 'total_errors', 'days_active', 'days_since_last_activity', 'usage_consistency_std', 'usage_trend', 'total_tickets', 'avg_resolution_hours', 'unresolved_tickets', 'technical_tickets_pct', 'high_priority_tickets', 'avg_satisfaction', 'days_since_last_ticket', 'churned']


---

## Step 2: Identify Categorical and Numeric Features

In [5]:
# Identify categorical columns (object dtype or specific columns)
categorical_cols = ml_data.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Identify numeric columns
numeric_cols = ml_data.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Remove target variable from numeric list
if "churned" in numeric_cols:
    numeric_cols.remove("churned")

print("Categorical columns:")
for col in categorical_cols:
    print(f"  - {col}: {ml_data[col].nunique()} unique values")

print(f"\nNumeric columns ({len(numeric_cols)}):")
print(numeric_cols)

print(f"\nTarget variable: churned")

Categorical columns:
  - gender: 4 unique values
  - subscription_tier: 3 unique values
  - age_group: 4 unique values
  - country_region: 4 unique values

Numeric columns (21):
['age', 'monthly_fee', 'account_age_days', 'is_premium', 'total_sessions', 'avg_sessions_per_day', 'total_usage_minutes', 'avg_session_duration', 'avg_features_per_session', 'total_errors', 'days_active', 'days_since_last_activity', 'usage_consistency_std', 'usage_trend', 'total_tickets', 'avg_resolution_hours', 'unresolved_tickets', 'technical_tickets_pct', 'high_priority_tickets', 'avg_satisfaction', 'days_since_last_ticket']

Target variable: churned


---

## Step 3: Encode Categorical Variables

### Strategy:
- **One-hot encoding** for: gender, subscription_tier, age_group, country_region
- This creates binary columns for each category
- We'll use `pd.get_dummies()` for simplicity

In [6]:
# Check unique values in categorical columns before encoding
print("Categorical variable distributions:")
print("=" * 60)
for col in categorical_cols:
    print(f"\n{col}:")
    print(ml_data[col].value_counts())

Categorical variable distributions:

gender:
gender
F          2332
M          2329
Other       189
Unknown     150
Name: count, dtype: int64

subscription_tier:
subscription_tier
Basic         3008
Premium       1476
Enterprise     516
Name: count, dtype: int64

age_group:
age_group
51+      1781
36-50    1593
26-35     918
18-25     708
Name: count, dtype: int64

country_region:
country_region
Europe           2477
North America    1765
Oceania           400
South America     358
Name: count, dtype: int64


In [7]:
# One-hot encode categorical variables
ml_data_encoded = pd.get_dummies(
    ml_data,
    columns=categorical_cols,
    drop_first=False,  # Keep all categories for interpretability
    dtype=int,  # Use int instead of bool for consistency
)

print(f"Shape before encoding: {ml_data.shape}")
print(f"Shape after encoding: {ml_data_encoded.shape}")
print(f"\nNew columns created: {ml_data_encoded.shape[1] - ml_data.shape[1]}")

Shape before encoding: (5000, 26)
Shape after encoding: (5000, 37)

New columns created: 11


In [8]:
# Show all column names after encoding
print("All columns after encoding:")
for i, col in enumerate(ml_data_encoded.columns, 1):
    print(f"{i:2d}. {col}")

All columns after encoding:
 1. age
 2. monthly_fee
 3. account_age_days
 4. is_premium
 5. total_sessions
 6. avg_sessions_per_day
 7. total_usage_minutes
 8. avg_session_duration
 9. avg_features_per_session
10. total_errors
11. days_active
12. days_since_last_activity
13. usage_consistency_std
14. usage_trend
15. total_tickets
16. avg_resolution_hours
17. unresolved_tickets
18. technical_tickets_pct
19. high_priority_tickets
20. avg_satisfaction
21. days_since_last_ticket
22. churned
23. gender_F
24. gender_M
25. gender_Other
26. gender_Unknown
27. subscription_tier_Basic
28. subscription_tier_Enterprise
29. subscription_tier_Premium
30. age_group_18-25
31. age_group_26-35
32. age_group_36-50
33. age_group_51+
34. country_region_Europe
35. country_region_North America
36. country_region_Oceania
37. country_region_South America


---

## Step 4: Handle Missing Values

Let's check if there are any remaining missing values

In [9]:
# Check for missing values
missing_values = ml_data_encoded.isna().sum()
missing_values = missing_values[missing_values > 0]

if len(missing_values) > 0:
    print("⚠️ Missing values found:")
    print(missing_values)
else:
    print("✅ No missing values in the dataset!")

✅ No missing values in the dataset!


---

## Step 5: Data Validation

Perform final quality checks before saving

### Check for potential data leakage

Make sure we don't have any features that would only be known AFTER the customer churns

In [10]:
print("Data Leakage Check")
print("=" * 60)
print("\n✓ No churn_date column (would be leakage)")
print("✓ No observation_date column (not needed)")
print("✓ All features are based on historical data before observation date")
print("\n✅ No data leakage detected!")

Data Leakage Check

✓ No churn_date column (would be leakage)
✓ No observation_date column (not needed)
✓ All features are based on historical data before observation date

✅ No data leakage detected!


---

## Step 6: Feature Summary

Let's document what features we have in the final dataset

In [11]:
# Separate features by type
feature_cols = [col for col in ml_data_encoded.columns if col != "churned"]

# Categorize features
demographic_features = [
    col for col in feature_cols if any(x in col for x in ["gender_", "age_group_"])
]
account_features = ["age", "monthly_fee", "account_age_days", "is_premium"] + [
    col for col in feature_cols if "subscription_tier_" in col
]
geographic_features = [col for col in feature_cols if "country_region_" in col]
usage_features = [
    col
    for col in feature_cols
    if any(
        x in col
        for x in [
            "session",
            "usage",
            "duration",
            "features",
            "errors",
            "activity",
            "consistency",
            "trend",
        ]
    )
]
support_features = [
    col
    for col in feature_cols
    if any(x in col for x in ["ticket", "resolution", "satisfaction", "priority"])
]

print("Feature Categories:")
print("=" * 60)
print(f"\nDemographic Features ({len(demographic_features)}):")
for feat in demographic_features:
    print(f"  - {feat}")

print(f"\nAccount Features ({len(account_features)}):")
for feat in account_features:
    print(f"  - {feat}")

print(f"\nGeographic Features ({len(geographic_features)}):")
for feat in geographic_features:
    print(f"  - {feat}")

print(f"\nUsage Features ({len(usage_features)}):")
for feat in usage_features:
    print(f"  - {feat}")

print(f"\nSupport Features ({len(support_features)}):")
for feat in support_features:
    print(f"  - {feat}")

print(f"\n{'='*60}")
print(f"Total Features: {len(feature_cols)}")
print(f"Target Variable: churned")

Feature Categories:

Demographic Features (8):
  - gender_F
  - gender_M
  - gender_Other
  - gender_Unknown
  - age_group_18-25
  - age_group_26-35
  - age_group_36-50
  - age_group_51+

Account Features (7):
  - age
  - monthly_fee
  - account_age_days
  - is_premium
  - subscription_tier_Basic
  - subscription_tier_Enterprise
  - subscription_tier_Premium

Geographic Features (4):
  - country_region_Europe
  - country_region_North America
  - country_region_Oceania
  - country_region_South America

Usage Features (9):
  - total_sessions
  - avg_sessions_per_day
  - total_usage_minutes
  - avg_session_duration
  - avg_features_per_session
  - total_errors
  - days_since_last_activity
  - usage_consistency_std
  - usage_trend

Support Features (7):
  - total_tickets
  - avg_resolution_hours
  - unresolved_tickets
  - technical_tickets_pct
  - high_priority_tickets
  - avg_satisfaction
  - days_since_last_ticket

Total Features: 36
Target Variable: churned


### Summary Statistics

In [12]:
# Show summary statistics for numeric features
ml_data_encoded.describe()

,age,monthly_fee,account_age_days,is_premium,total_sessions,avg_sessions_per_day,total_usage_minutes,avg_session_duration,avg_features_per_session,total_errors,days_active,days_since_last_activity,usage_consistency_std,usage_trend,total_tickets,avg_resolution_hours,unresolved_tickets,technical_tickets_pct,high_priority_tickets,avg_satisfaction,days_since_last_ticket,churned,gender_F,gender_M,gender_Other,gender_Unknown,subscription_tier_Basic,subscription_tier_Enterprise,subscription_tier_Premium,age_group_18-25,age_group_26-35,age_group_36-50,age_group_51+,country_region_Europe,country_region_North America,country_region_Oceania,country_region_South America
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,43.94,33.48,635.96,0.40,213.01,5.00,1267.20,29.75,7.49,21.30,42.59,2.06,2.57,-0.00,1.19,6.10,0.24,20.08,0.19,1.22,42.13,0.25,0.47,0.47,0.04,0.03,0.60,0.10,0.30,0.14,0.18,0.32,0.36,0.50,0.35,0.08,0.07
std,14.90,42.24,260.07,0.49,110.65,0.49,661.99,4.00,0.78,11.85,21.83,4.01,0.24,1.00,1.70,9.35,0.56,32.69,0.48,1.86,75.58,0.43,0.50,0.50,0.19,0.17,0.49,0.30,0.46,0.35,0.39,0.47,0.48,0.50,0.48,0.27,0.26
min,18.00,9.99,184.00,0.00,15.00,2.50,54.70,9.12,2.00,0.00,5.00,0.00,0.82,-5.14,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,32.00,12.11,410.75,0.00,124.00,4.71,749.70,27.34,7.05,12.00,25.00,0.00,2.45,-0.58,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,43.50,14.19,635.00,0.00,209.00,5.00,1239.70,29.66,7.48,20.00,42.00,1.00,2.58,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,57.00,35.24,855.00,1.00,285.00,5.29,1709.34,32.03,7.94,29.00,57.00,2.00,2.70,0.58,2.00,11.86,0.00,40.00,0.00,3.00,56.00,0.25,1.00,1.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00
max,70.00,199.97,1095.00,1.00,513.00,8.00,3385.77,64.58,12.17,59.00,89.00,47.00,3.79,5.50,5.00,82.01,4.00,100.00,4.00,5.00,365.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00


---

## Step 7: Save Final ML-Ready Dataset

In [13]:
# Save to final folder
ml_data_encoded.to_csv("../final/ml_ready_dataset.csv", index=False)

print("✅ Saved ml_ready_dataset.csv to final folder")
print(f"   Shape: {ml_data_encoded.shape}")
print(f"   Features: {len(feature_cols)}")
print(f"   Target: churned")
print(
    f"   File size: {ml_data_encoded.memory_usage(deep=True).sum() / 1024**2:.2f} MB in memory"
)

✅ Saved ml_ready_dataset.csv to final folder
   Shape: (5000, 37)
   Features: 36
   Target: churned
   File size: 1.41 MB in memory


---

## Step 8: Demonstrate Train/Test Split

Show how to properly split this data for machine learning

In [14]:
print("Train/Test Split Demonstration")
print("=" * 60)

# Separate features and target
X = ml_data_encoded.drop("churned", axis=1)
y = ml_data_encoded["churned"]

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"Churn rate: {y.mean():.2%}")

Train/Test Split Demonstration
Features (X): (5000, 36)
Target (y): (5000,)

Target distribution:
churned
0    3750
1    1250
Name: count, dtype: int64
Churn rate: 25.00%


In [15]:
# Perform stratified train/test split
# stratify=y ensures the same churn rate in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # 20% for testing
    random_state=42,  # For reproducibility
    stratify=y,  # Maintain churn rate in both sets
)

print("\nTrain/Test Split Results:")
print("=" * 60)
print(f"\nTraining Set:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  Churn rate: {y_train.mean():.2%}")
print(f"  Class distribution:\n{y_train.value_counts()}")

print(f"\nTest Set:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")
print(f"  Churn rate: {y_test.mean():.2%}")
print(f"  Class distribution:\n{y_test.value_counts()}")

print(f"\n✅ Churn rate is balanced between train and test sets!")


Train/Test Split Results:

Training Set:
  X_train shape: (4000, 36)
  y_train shape: (4000,)
  Churn rate: 25.00%
  Class distribution:
churned
0    3000
1    1000
Name: count, dtype: int64

Test Set:
  X_test shape: (1000, 36)
  y_test shape: (1000,)
  Churn rate: 25.00%
  Class distribution:
churned
0    750
1    250
Name: count, dtype: int64

✅ Churn rate is balanced between train and test sets!


### Important Notes on Feature Scaling

⚠️ **Feature scaling should be done AFTER train/test split!**

Here's why and how:

In [16]:
print("Feature Scaling Guidelines")
print("=" * 60)
print("\n✅ DO scale these features (continuous numeric):")
continuous_features = [
    "age",
    "monthly_fee",
    "account_age_days",
    "total_sessions",
    "avg_sessions_per_day",
    "total_usage_minutes",
    "avg_session_duration",
    "avg_features_per_session",
    "days_active",
    "days_since_last_activity",
    "usage_consistency_std",
    "total_tickets",
    "avg_resolution_hours",
    "days_since_last_ticket",
]
for feat in continuous_features:
    if feat in X.columns:
        print(f"  - {feat}")

print("\n❌ DON'T scale these features (binary/one-hot encoded):")
binary_features = [
    col
    for col in X.columns
    if col.startswith(
        ("gender_", "subscription_tier_", "age_group_", "country_region_")
    )
    or col in ["is_premium", "unresolved_tickets", "high_priority_tickets"]
]
for feat in binary_features[:10]:  # Show first 10
    print(f"  - {feat}")
print(f"  ... and {len(binary_features) - 10} more")

print("\n💡 Example scaling code (to be done AFTER split):")
print("```python")
print("from sklearn.preprocessing import StandardScaler")
print("")
print("scaler = StandardScaler()")
print("X_train_scaled = scaler.fit_transform(X_train[continuous_features])")
print(
    "X_test_scaled = scaler.transform(X_test[continuous_features])  # Use same scaler!"
)
print("```")

Feature Scaling Guidelines

✅ DO scale these features (continuous numeric):
  - age
  - monthly_fee
  - account_age_days
  - total_sessions
  - avg_sessions_per_day
  - total_usage_minutes
  - avg_session_duration
  - avg_features_per_session
  - days_active
  - days_since_last_activity
  - usage_consistency_std
  - total_tickets
  - avg_resolution_hours
  - days_since_last_ticket

❌ DON'T scale these features (binary/one-hot encoded):
  - is_premium
  - unresolved_tickets
  - high_priority_tickets
  - gender_F
  - gender_M
  - gender_Other
  - gender_Unknown
  - subscription_tier_Basic
  - subscription_tier_Enterprise
  - subscription_tier_Premium
  ... and 8 more

💡 Example scaling code (to be done AFTER split):
```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[continuous_features])
X_test_scaled = scaler.transform(X_test[continuous_features])  # Use same scaler!
```


---

## Summary

### What We Accomplished in the Final Layer:

✅ **Feature Selection**:
- Removed identifier columns (customer_id)
- Removed redundant columns (signup_date, country)
- Kept all predictive features

✅ **Encoding**:
- One-hot encoded: gender, subscription_tier, age_group, country_region
- Result: All features are now numeric

✅ **Validation**:
- No missing values
- No infinite values
- No data leakage
- No duplicate rows
- Balanced target distribution in train/test

✅ **Documentation**:
- Categorized features by type
- Identified which features to scale
- Demonstrated proper train/test split

### Final Dataset:
- **Rows**: 5,000 customers
- **Features**: ~30+ features (depending on encoding)
- **Target**: churned (binary: 0 or 1)
- **Churn Rate**: ~25%

### Next Steps for ML:

1. **Load the final dataset**
2. **Split into train/test** (as demonstrated above)
3. **Scale continuous features** (using training set statistics)
4. **Train models** (Logistic Regression, Random Forest, XGBoost, etc.)
5. **Evaluate performance** (accuracy, precision, recall, F1, ROC-AUC)
6. **Tune hyperparameters**
7. **Make predictions**

### Key Learnings:

1. **ETL Pipeline** - Raw → Staging → Intermediate → Final
2. **Separation of Concerns** - Each layer has a specific purpose
3. **Feature Engineering** - Domain knowledge creates better features
4. **Data Quality** - Validate at every step
5. **No Data Leakage** - Only use information available at prediction time
6. **Proper Splitting** - Stratify to maintain class balance
7. **Scale After Split** - Prevent data leakage from scaling

🎉 **Congratulations! You've completed the entire ETL pipeline!**